In [33]:
# !pip install evaluate

In [59]:
import pandas as pd
import tensorflow as tf
import torch, evaluate
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
        BertTokenizer, BertForSequenceClassification,
          TrainingArguments, Trainer, pipeline)


In [35]:
df = pd.read_csv('SMSSpamCollection', sep='\t',
                 names= ['label', 'text'])

In [36]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [37]:
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [38]:
train, test = train_test_split(df, test_size=0.2, random_state=0)

### Convert Pandas df to huggingface datasets

In [39]:
train_dataset = Dataset.from_pandas(train)
test_dataset = Dataset.from_pandas(test)

In [40]:
type(train_dataset)

datasets.arrow_dataset.Dataset

### Tokenization

In [41]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [42]:
def tokenizer_function(examples):
  return tokenizer(examples['text'],
                  padding='max_length',truncation=True,
                  max_length=128)

### Apply the tokenizer to datasets

In [43]:
train_dataset = train_dataset.map(tokenizer_function, batched=True)
test_dataset = test_dataset.map(tokenizer_function, batched=True)

Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

### Remove the non-tensor column

In [44]:
train_dataset = train_dataset.remove_columns(['text', '__index_level_0__'])
test_dataset = test_dataset.remove_columns(['text','__index_level_0__'])

### Load the Model

In [45]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### define Evaluation Metric

In [46]:
metric = evaluate.load("accuracy")

In [47]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

###

In [48]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy='epoch',
    save_strategy='epoch'

)   ## Auto Generated first 4 atleast imp

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Define Trainer

In [49]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [50]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.034164,0.990135
2,0.116380,0.043315,0.992825
3,0.116380,0.021682,0.995516


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=837, training_loss=0.07928485027208408, metrics={'train_runtime': 366.5288, 'train_samples_per_second': 36.48, 'train_steps_per_second': 2.284, 'total_flos': 879514480304640.0, 'train_loss': 0.07928485027208408, 'epoch': 3.0})

### Save the Model

In [52]:
model.save_pretrained('./model')
tokenizer.save_pretrained('./tokenizer')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./tokenizer/tokenizer_config.json', './tokenizer/tokenizer.json')

### Load The Model


In [60]:
spam_classifier = pipeline(
    'text-classification',
    model= './model',
    tokenizer= './tokenizer'
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [68]:
new_messages = [
    'Hey Rahul, are you still in the meeting? Let us meet at lunch today.',
    'Urgent! You have been selected to recieve a $900 prize reward! Reply earliest'
]

In [69]:
predictions = spam_classifier(new_messages)
print(predictions)

[{'label': 'LABEL_0', 'score': 0.9997225403785706}, {'label': 'LABEL_1', 'score': 0.999670147895813}]
